In [2]:
import pandas as pd
import numpy as np


def binary_cross_entropy(p, q, epsilon=1e-12):
    """
    Binary cross entropy between two probability arrays.
    """
    p = np.clip(p, epsilon, 1 - epsilon)
    q = np.clip(q, epsilon, 1 - epsilon)

    return -np.mean(
        p * np.log(q) +
        (1 - p) * np.log(1 - q)
    )


# ======================================================
# SETTINGS
# ======================================================

# CSV file names (same folder as notebook)
nn_file = "classification_NN_rough.csv"
bdt_file = "classification_XGB_rough.csv"

# Column names
nn_col = "prediction"
bdt_col = "probability"

# ======================================================
# LOAD DATA
# ======================================================

nn_df = pd.read_csv(nn_file)
bdt_df = pd.read_csv(bdt_file)

print("NN columns:")
print(nn_df.columns.tolist())

print("\nBDT columns:")
print(bdt_df.columns.tolist())

# Extract probability scores
nn_scores = nn_df[nn_col].to_numpy(dtype=float)
bdt_scores = bdt_df[bdt_col].to_numpy(dtype=float)

# ======================================================
# CHECKS
# ======================================================

if len(nn_scores) != len(bdt_scores):
    raise ValueError(
        f"Different lengths: {len(nn_scores)} vs {len(bdt_scores)}"
    )

if np.any((nn_scores < 0) | (nn_scores > 1)):
    raise ValueError("NN scores are not all in [0,1]")

if np.any((bdt_scores < 0) | (bdt_scores > 1)):
    raise ValueError("BDT scores are not all in [0,1]")

# ======================================================
# METRICS
# ======================================================

ce_nn_bdt = binary_cross_entropy(nn_scores, bdt_scores)
ce_bdt_nn = binary_cross_entropy(bdt_scores, nn_scores)

symmetric_ce = 0.5 * (ce_nn_bdt + ce_bdt_nn)

correlation = np.corrcoef(nn_scores, bdt_scores)[0, 1]

mean_abs_diff = np.mean(np.abs(nn_scores - bdt_scores))

# ======================================================
# OUTPUT
# ======================================================

print("\n===== RESULTS =====")
print(f"Cross Entropy H(NN,BDT): {ce_nn_bdt:.6f}")
print(f"Cross Entropy H(BDT,NN): {ce_bdt_nn:.6f}")
print(f"Symmetric Cross Entropy: {symmetric_ce:.6f}")

print(f"\nPearson Correlation: {correlation:.6f}")
print(f"Mean Absolute Difference: {mean_abs_diff:.6f}")

print("\nInterpretation:")
print("- Lower cross entropy => more similar outputs")
print("- Correlation near 1 => strong agreement")
print("- Mean absolute difference near 0 => predictions are very close")

NN columns:
['prediction', 'class']

BDT columns:
['prediction', 'probability']

===== RESULTS =====
Cross Entropy H(NN,BDT): 0.168010
Cross Entropy H(BDT,NN): 0.113069
Symmetric Cross Entropy: 0.140540

Pearson Correlation: 0.978015
Mean Absolute Difference: 0.029074

Interpretation:
- Lower cross entropy => more similar outputs
- Correlation near 1 => strong agreement
- Mean absolute difference near 0 => predictions are very close
